## 7. Drug-Drug Interaction (DDI) Analysis

> **Note on data scope:** Sections 1–6 use Q2 2025 data only (course project EDA and ML modeling). Section 7 loads all four quarters (Q1–Q4 2025, ~1.47M deduplicated reports) for the thesis DDI pharmacovigilance analysis. The larger multi-quarter dataset is necessary to accumulate sufficient signal counts for rare drug combinations.

This section extends the analysis to detect potential drug-drug interactions using pharmacovigilance methods. We follow FDA guidelines for signal detection using Reporting Odds Ratios (ROR).

### 7.1 Multi-Quarter Data Loading

Load 4 quarters of FAERS 2025 data (Q1-Q4) to ensure sufficient sample sizes for drug combination analysis.

## Prerequisites

**DDI pipeline entry point — starts from raw FAERS data.**

This notebook covers Sections 7.1-7.4 of the original thesis work: loading all 4 quarters of FAERS, standardizing drug names via RxNorm, and extracting drug pairs.

**Data required:**
- Raw FAERS ASCII files at `/Users/joshbuck/Downloads/faers_ascii_2025qX/ASCII/` for X in {1, 2, 3, 4}
- `../data/cache/rxnorm_mapping_cache.json` — speeds up RxNorm lookups (DO NOT DELETE)

**Outputs (for downstream notebooks):**
- `../data/intermediate/FAERS_DRUG_PAIRS_2025.csv` — raw drug pairs (pre-RxNorm)
- `../data/intermediate/FAERS_DRUG_PAIRS_RXNORM.csv` — standardized pairs (PRIMARYID + DRUG_A + DRUG_B + PAIR + SERIOUS)
- `../data/intermediate/rxnorm_drug_mapping.csv` — name mapping

**Kernel variables produced** (used by notebooks 03, 06):
- `demo_all` — demographics across all quarters
- `outcome_flags` — PRIMARYID + SERIOUS for all patients
- `drug_pairs_std_filtered` — RxNorm-standardized pairs with 10+ occurrences

**⚠️ If you don't have the raw FAERS files, you cannot run this notebook end-to-end.** The intermediate CSVs in `../data/intermediate/` were produced by a previous successful run.

In [1]:
import pandas as pd
import numpy as np
import os
import json
import time
import random
import requests
from itertools import combinations

/Users/joshbuck/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# =============================================================================
# STEP 1: MULTI-QUARTER LOADING + RXNORM STANDARDIZATION + DRUG PAIRS
# =============================================================================
# 1. Load 4 quarters of FAERS data
# 2. Standardize drug names using RxNorm
# 3. Extract two-drug combinations per patient
# 4. Keep combinations occurring 10+ times


# -----------------------------------------------------------------------------
# PART 1: Load all 4 quarters
# -----------------------------------------------------------------------------
base_path = '/Users/joshbuck/Downloads/'  # Update this path for your machine

quarters = {
    'Q1_2025': 'faers_ascii_2025q1/ASCII/DRUG25Q1.txt',
    'Q2_2025': 'faers_ascii_2025q2/ASCII/DRUG25Q2.txt',
    'Q3_2025': 'faers_ascii_2025q3/ASCII/DRUG25Q3.txt',
    'Q4_2025': 'faers_ascii_2025Q4/ASCII/DRUG25Q4.txt',
}

demo_files = {
    'Q1_2025': 'faers_ascii_2025q1/ASCII/DEMO25Q1.txt',
    'Q2_2025': 'faers_ascii_2025q2/ASCII/DEMO25Q2.txt',
    'Q3_2025': 'faers_ascii_2025q3/ASCII/DEMO25Q3.txt',
    'Q4_2025': 'faers_ascii_2025Q4/ASCII/DEMO25Q4.txt',
}

outc_files = {
    'Q1_2025': 'faers_ascii_2025q1/ASCII/OUTC25Q1.txt',
    'Q2_2025': 'faers_ascii_2025q2/ASCII/OUTC25Q2.txt',
    'Q3_2025': 'faers_ascii_2025q3/ASCII/OUTC25Q3.txt',
    'Q4_2025': 'faers_ascii_2025Q4/ASCII/OUTC25Q4.txt',
}

print("Loading DRUG tables from all 4 quarters...")
print("=" * 60)

drug_dfs = []
for quarter, filepath in quarters.items():
    full_path = base_path + filepath
    df = pd.read_csv(full_path, delimiter='$', encoding='utf-8', low_memory=False)
    df.columns = df.columns.str.strip().str.upper()
    df['QUARTER'] = quarter
    drug_dfs.append(df)
    print(f"  {quarter}: {len(df):,} drug records")

drug_all = pd.concat(drug_dfs, ignore_index=True)
print(f"\nTotal DRUG records: {len(drug_all):,}")

print("\nLoading DEMO tables...")
demo_dfs = []
for quarter, filepath in demo_files.items():
    full_path = base_path + filepath
    df = pd.read_csv(full_path, delimiter='$', encoding='utf-8', low_memory=False)
    df.columns = df.columns.str.strip().str.upper()
    df['QUARTER'] = quarter
    demo_dfs.append(df)
    print(f"  {quarter}: {len(df):,} reports")

demo_all = pd.concat(demo_dfs, ignore_index=True)
print(f"Total DEMO records: {len(demo_all):,}")

print("\nLoading OUTC tables...")
outc_dfs = []
for quarter, filepath in outc_files.items():
    full_path = base_path + filepath
    df = pd.read_csv(full_path, delimiter='$', encoding='utf-8', low_memory=False)
    df.columns = df.columns.str.strip().str.upper()
    outc_dfs.append(df)
    print(f"  {quarter}: {len(df):,} outcomes")

outc_all = pd.concat(outc_dfs, ignore_index=True)
print(f"Total OUTC records: {len(outc_all):,}")

print("\n" + "=" * 60)
print("ALL QUARTERS LOADED SUCCESSFULLY")
print("=" * 60)

Loading DRUG tables from all 4 quarters...
  Q1_2025: 2,008,162 drug records
  Q2_2025: 1,829,056 drug records
  Q3_2025: 2,148,451 drug records
  Q4_2025: 1,815,349 drug records

Total DRUG records: 7,801,018

Loading DEMO tables...
  Q1_2025: 400,514 reports
  Q2_2025: 393,130 reports
  Q3_2025: 438,512 reports
  Q4_2025: 385,288 reports
Total DEMO records: 1,617,444

Loading OUTC tables...
  Q1_2025: 304,027 outcomes
  Q2_2025: 295,583 outcomes
  Q3_2025: 343,251 outcomes
  Q4_2025: 289,721 outcomes
Total OUTC records: 1,232,582

ALL QUARTERS LOADED SUCCESSFULLY


In [3]:
# =============================================================================
# CASEID DEDUPLICATION
# =============================================================================
# FAERS contains duplicate reports for the same case (follow-ups, corrections).
# Each case has a unique CASEID but may have multiple PRIMARYIDs.
# Standard practice: keep only the most recent report per CASEID
# (highest PRIMARYID = latest version with most complete info).

print("CASEID DEDUPLICATION")
print("=" * 60)

print(f"DEMO records before dedup: {len(demo_all):,}")
print(f"Unique CASEIDs: {demo_all['CASEID'].nunique():,}")
print(f"Unique PRIMARYIDs: {demo_all['PRIMARYID'].nunique():,}")
print(f"Duplicate reports: {len(demo_all) - demo_all['CASEID'].nunique():,}")

# Keep only the row with the highest PRIMARYID per CASEID
demo_all = demo_all.sort_values('PRIMARYID', ascending=False)
demo_all = demo_all.drop_duplicates(subset='CASEID', keep='first')

print(f"\nDEMO records after dedup: {len(demo_all):,}")
print(f"Removed: {len(demo_dfs[0]) + len(demo_dfs[1]) + len(demo_dfs[2]) + len(demo_dfs[3]) - len(demo_all):,} duplicate reports")

# Filter DRUG and OUTC tables to only keep deduplicated PRIMARYIDs
valid_primaryids = set(demo_all['PRIMARYID'].tolist())

drug_before = len(drug_all)
drug_all = drug_all[drug_all['PRIMARYID'].isin(valid_primaryids)]
print(f"\nDRUG records: {drug_before:,} -> {len(drug_all):,}")

outc_before = len(outc_all)
outc_all = outc_all[outc_all['PRIMARYID'].isin(valid_primaryids)]
print(f"OUTC records: {outc_before:,} -> {len(outc_all):,}")

print(f"\nDeduplication complete. All downstream analysis uses unique cases only.")

CASEID DEDUPLICATION
DEMO records before dedup: 1,617,444
Unique CASEIDs: 1,469,305
Unique PRIMARYIDs: 1,617,313
Duplicate reports: 148,139

DEMO records after dedup: 1,469,305
Removed: 148,139 duplicate reports

DRUG records: 7,801,018 -> 6,645,494
OUTC records: 1,232,582 -> 1,099,691

Deduplication complete. All downstream analysis uses unique cases only.


### 7.2 RxNorm Drug Name Standardization

Standardize drug names using NIH RxNorm API to merge brand names, generics, and misspellings. This uses RXCUI codes as unique identifiers.

In [4]:
# =============================================================================
# RXNORM DRUG NAME STANDARDIZATION
# =============================================================================
# Uses NIH RxNorm API to map drug names to standardized RxCUI codes
# Results are saved to cache so you only run this once


print("RXNORM DRUG NAME STANDARDIZATION")
print("=" * 60)

CACHE_FILE = "../data/cache/rxnorm_mapping_cache.json"

# Check if cache exists - if so, just load it and skip API calls
if os.path.exists(CACHE_FILE):
    print("Loading RxNorm mappings from cache...")
    with open(CACHE_FILE, 'r') as f:
        rxnorm_cache = json.load(f)
    print(f" Loaded {len(rxnorm_cache):,} cached mappings - SKIPPING API calls")

    # Create mapping DataFrame from cache
    rxnorm_df = pd.DataFrame([
        {'DRUGNAME_CLEAN': drug, **info}
        for drug, info in rxnorm_cache.items()
    ])
    rxnorm_df.to_csv("../data/intermediate/rxnorm_drug_mapping.csv", index=False)

    total_mapped = sum(1 for v in rxnorm_cache.values() if v.get('RXCUI'))
    print(f"Successfully mapped: {total_mapped:,} ({100*total_mapped/len(rxnorm_cache):.1f}%)")
    print(f"Ready to proceed with standardized drug names")

else:
    # No cache - need to run full API mapping
    print("No cache found - running full RxNorm API mapping...")

    # Get unique drug names to map
    unique_drugs = drug_clean['DRUGNAME_CLEAN'].unique().tolist()
    print(f"Unique drug names to map: {len(unique_drugs):,}")

    rxnorm_cache = {}

    def get_rxnorm_info(drug_name, max_retries=3):
        base_url = "https://rxnav.nlm.nih.gov/REST"
        for attempt in range(max_retries):
            try:
                url = f"{base_url}/approximateTerm.json?term={requests.utils.quote(drug_name)}&maxEntries=1"
                response = requests.get(url, timeout=10)
                if response.status_code == 200:
                    data = response.json()
                    if 'approximateGroup' in data and 'candidate' in data['approximateGroup']:
                        candidates = data['approximateGroup']['candidate']
                        if candidates and len(candidates) > 0:
                            best_match = candidates[0]
                            return {
                                'RXCUI': best_match.get('rxcui'),
                                'RXNORM_NAME': best_match.get('name', '').upper(),
                                'SCORE': best_match.get('score'),
                                'RANK': best_match.get('rank')
                            }
                    return {'RXCUI': None, 'RXNORM_NAME': None, 'SCORE': None, 'RANK': None}
                elif response.status_code == 429:
                    time.sleep(2)
                    continue
            except:
                time.sleep(1)
                continue
        return {'RXCUI': None, 'RXNORM_NAME': None, 'SCORE': None, 'RANK': None}

    print("This will take 2-4 hours. Progress saved every 100 drugs.")

    start_time = time.time()
    for i, drug in enumerate(unique_drugs):
        if drug not in rxnorm_cache:
            rxnorm_cache[drug] = get_rxnorm_info(drug)

        if (i + 1) % 100 == 0:
            elapsed = time.time() - start_time
            rate = (i + 1) / elapsed
            remaining = len(unique_drugs) - (i + 1)
            eta_minutes = remaining / rate / 60
            mapped_count = sum(1 for v in rxnorm_cache.values() if v.get('RXCUI'))
            print(f"  Progress: {i+1:,}/{len(unique_drugs):,} ({100*(i+1)/len(unique_drugs):.1f}%) | Mapped: {mapped_count:,} | ETA: {eta_minutes:.0f} min")
            with open(CACHE_FILE, 'w') as f:
                json.dump(rxnorm_cache, f)

        time.sleep(0.1)

    # Final save
    with open(CACHE_FILE, 'w') as f:
        json.dump(rxnorm_cache, f)

    rxnorm_df = pd.DataFrame([
        {'DRUGNAME_CLEAN': drug, **info}
        for drug, info in rxnorm_cache.items()
    ])
    rxnorm_df.to_csv("../data/intermediate/rxnorm_drug_mapping.csv", index=False)

    total_mapped = sum(1 for v in rxnorm_cache.values() if v.get('RXCUI'))
    print(f"\n COMPLETE: {total_mapped:,}/{len(rxnorm_cache):,} mapped ({100*total_mapped/len(rxnorm_cache):.1f}%)")

RXNORM DRUG NAME STANDARDIZATION
Loading RxNorm mappings from cache...
 Loaded 64,552 cached mappings - SKIPPING API calls
Successfully mapped: 59,316 (91.9%)
Ready to proceed with standardized drug names


### 7.3 Drug Pair Extraction

Extract all two-drug combinations per patient report. Filter to combinations occurring 10+ times for statistical reliability.

In [5]:
# -----------------------------------------------------------------------------
# PART 2: CLEAN AND STANDARDIZE DRUG NAMES
# -----------------------------------------------------------------------------
print("PART 2: Standardizing drug names...")
print("=" * 60)

import random

# Check what drug roles exist
print("Drug roles in dataset:")
print(drug_all['ROLE_COD'].value_counts())

# For DDI analysis, we need ALL drugs the patient was taking, not just PS(primary suspect)
# ROLE_COD values: PS=Primary Suspect, SS=Secondary Suspect, C=Concomitant, I=Interacting
# Keep all drugs to find combinations

drug_clean = drug_all.copy()
print(f"\nUsing ALL drugs for DDI analysis: {len(drug_clean):,} records")

# Basic drug name standardization
def standardize_drug_name(name):
    if pd.isna(name):
        return None
    name = str(name).upper().strip()
    name = name.replace('.', '').replace(',', '')
    if '(' in name:
        name = name.split('(')[0].strip()
    return name if name else None

drug_clean['DRUGNAME_CLEAN'] = drug_clean['DRUGNAME'].apply(standardize_drug_name)
drug_clean = drug_clean[drug_clean['DRUGNAME_CLEAN'].notna()]

print(f"After cleaning: {len(drug_clean):,} drug records")
print(f"Unique drug names (standardized): {drug_clean['DRUGNAME_CLEAN'].nunique():,}")

#show top 20 drugs
print("\nTop 20 Drugs (all roles):")
print(drug_clean['DRUGNAME_CLEAN'].value_counts().head(20))

# -----------------------------------------------------------------------------
# PART 3: CREATE SERIOUS OUTCOME FLAG
# -----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("PART 3: Creating serious outcome flags...")

serious_codes = ['DE', 'LT', 'HO', 'DS', 'CA', 'RI']
outc_all['SERIOUS'] = outc_all['OUTC_COD'].isin(serious_codes).astype(int)
outcome_flags = outc_all.groupby('PRIMARYID')['SERIOUS'].max().reset_index()
print(f"Patients with outcomes: {len(outcome_flags):,}")

# -----------------------------------------------------------------------------
# PART 4: EXTRACT DRUG PAIRS PER PATIENT
# -----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("PART 4: Extracting drug pairs per patient...")

# Get unique drugs per patient
patient_drugs = drug_clean.groupby('PRIMARYID')['DRUGNAME_CLEAN'].apply(lambda x: list(set(x))).reset_index()
patient_drugs['NUM_DRUGS'] = patient_drugs['DRUGNAME_CLEAN'].apply(len)

print(f"Total patients: {len(patient_drugs):,}")
print(f"\nDrugs per patient distribution:")
print(patient_drugs['NUM_DRUGS'].describe())
print(f"\nPatients by drug count:")
print(patient_drugs['NUM_DRUGS'].value_counts().head(10))

# Filter to patients with 2+ unique drugs
multi_drug_patients = patient_drugs[patient_drugs['NUM_DRUGS'] >= 2].copy()
print(f"\nPatients with 2+ unique drugs: {len(multi_drug_patients):,}")

if len(multi_drug_patients) == 0:
    print("ERROR: No patients with 2+ drugs found!")
else:
    # Extract drug pairs
    print("\nExtracting drug pairs...")


    drug_pairs = []
    total = len(multi_drug_patients)

    for i, (idx, row) in enumerate(multi_drug_patients.iterrows()):
        if i % 100000 == 0:
            print(f"  Processing: {i:,}/{total:,} patients ({100*i/total:.1f}%)")

        primaryid = row['PRIMARYID']
        drugs = sorted(row['DRUGNAME_CLEAN'])  # Already unique from set()

        # Cap at 20 drugs per pateint, random sample
        if len(drugs) > 20:
            drugs = sorted(random.sample(drugs, 20))

        for pair in combinations(drugs, 2):
            drug_pairs.append((primaryid, pair[0], pair[1], f"{pair[0]} + {pair[1]}"))

    print(f"  Processing: {total:,}/{total:,} patients (100%)")

    # Create df
    drug_pairs_df = pd.DataFrame(drug_pairs, columns=['PRIMARYID', 'DRUG_A', 'DRUG_B', 'PAIR'])
    print(f"\nTotal drug pairs extracted: {len(drug_pairs_df):,}")
    print(f"Unique drug combinations: {drug_pairs_df['PAIR'].nunique():,}")

    # -----------------------------------------------------------------------------
    # PART 5: FILTER TO COMBINATIONS WITH 10+ OCCURRENCES
    # -----------------------------------------------------------------------------
    print("\n" + "=" * 60)
    print("PART 5: Filtering to combinations with 10+ occurrences...")

    pair_counts = drug_pairs_df['PAIR'].value_counts()
    print(f"Pair frequency distribution:")
    print(f"  Pairs appearing 1 time: {(pair_counts == 1).sum():,}")
    print(f"  Pairs appearing 2-9 times: {((pair_counts >= 2) & (pair_counts < 10)).sum():,}")
    print(f"  Pairs appearing 10+ times: {(pair_counts >= 10).sum():,}")
    print(f"  Pairs appearing 100+ times: {(pair_counts >= 100).sum():,}")

    # Keep only pairs with 10+ occurrences
    frequent_pairs = pair_counts[pair_counts >= 10].index.tolist()
    drug_pairs_filtered = drug_pairs_df[drug_pairs_df['PAIR'].isin(frequent_pairs)].copy()

    print(f"\nAfter filtering to 10+ occurrences:")
    print(f"  Drug pairs remaining: {len(drug_pairs_filtered):,}")
    print(f"  Unique combinations: {drug_pairs_filtered['PAIR'].nunique():,}")

    # Merge with outcome data
    drug_pairs_filtered = drug_pairs_filtered.merge(outcome_flags, on='PRIMARYID', how='inner')
    # No fillna needed — inner join means every patient has an outcome record
    print(f"Patients with outcome data retained: {drug_pairs_filtered['PRIMARYID'].nunique():,}")

    print(f"\nSerious outcomes in filtered pairs:")
    print(drug_pairs_filtered['SERIOUS'].value_counts())

    # -----------------------------------------------------------------------------
    # SUMMARY
    # -----------------------------------------------------------------------------
    print("\n" + "=" * 60)
    print("STEP 1 COMPLETE - SUMMARY")
    print("=" * 60)
    print(f"Quarters loaded: 4 (Q1-Q4 2025)")
    print(f"Total patient reports: {len(demo_all):,}")
    print(f"Patients with 2+ drugs: {len(multi_drug_patients):,}")
    print(f"Drug pairs (10+ occurrences): {drug_pairs_filtered['PAIR'].nunique():,}")
    print(f"Total pair-patient observations: {len(drug_pairs_filtered):,}")

    print("\nTop 20 Most Common Drug Pairs:")
    print(drug_pairs_filtered['PAIR'].value_counts().head(20))

    # Save for next steps
    drug_pairs_filtered.to_csv("../data/intermediate/FAERS_DRUG_PAIRS_2025.csv", index=False)
    print("\n Saved: FAERS_DRUG_PAIRS_2025.csv")

PART 2: Standardizing drug names...
Drug roles in dataset:
ROLE_COD
SS    2570520
C     2446234
PS    1593446
I       35111
DN        183
Name: count, dtype: int64

Using ALL drugs for DDI analysis: 6,645,494 records
After cleaning: 6,645,476 drug records
Unique drug names (standardized): 64,156

Top 20 Drugs (all roles):
DRUGNAME_CLEAN
DUPIXENT         173406
MOUNJARO         119587
ZEPBOUND         110925
PREDNISONE        89458
METHOTREXATE      78862
RITUXIMAB         62565
INFLECTRA         58088
VEDOLIZUMAB       52960
ACETAMINOPHEN     50352
ACTEMRA           46186
FOLIC ACID        43415
INFLIXIMAB        41428
ASPIRIN           41030
SULFASALAZINE     39149
DEXAMETHASONE     30478
ATORVASTATIN      29937
ORENCIA           29268
HUMIRA            28704
OZEMPIC           28410
SKYRIZI           28104
Name: count, dtype: int64

PART 3: Creating serious outcome flags...
Patients with outcomes: 823,213

PART 4: Extracting drug pairs per patient...
Total patients: 1,469,305

Drugs p

NameError: name 'combinations' is not defined

### 7.4 Apply RxNorm Standardization to Drug Pairs

Apply the RXCUI mappings to further deduplicate drug pairs and re-extract combinations.

In [ ]:
# =============================================================================
# APPLY RXNORM STANDARDIZATION USING RXCUI CODES
# =============================================================================
# Using RXCUI (unique identifier) instead of names for better deduplication
# Multiple drug names can map to the same RXCUI

print("Applying RxNorm standardization using RXCUI codes...")
print("=" * 60)

# Load the RxNorm mapping
rxnorm_map = pd.read_csv("../data/intermediate/rxnorm_drug_mapping.csv")
print(f"RxNorm mappings loaded: {len(rxnorm_map):,}")
print(f"Successfully mapped to RXCUI: {rxnorm_map['RXCUI'].notna().sum():,}")

# Create mapping dictionaries
# 1. Original name -> RXCUI (for grouping)
# 2. RXCUI -> Standardized name (for display)

name_to_rxcui = {}
rxcui_to_name = {}

for _, row in rxnorm_map.iterrows():
    original = row['DRUGNAME_CLEAN']
    rxcui = row['RXCUI']
    rxnorm_name = row['RXNORM_NAME']

    if pd.notna(rxcui):
        name_to_rxcui[original] = str(int(float(rxcui)))  # Use RXCUI as string
        if str(int(float(rxcui))) not in rxcui_to_name and pd.notna(rxnorm_name):
            rxcui_to_name[str(int(float(rxcui)))] = rxnorm_name.upper()
    else:
        # No RXCUI - use original name as identifier
        name_to_rxcui[original] = f"UNMAPPED_{original}"
        rxcui_to_name[f"UNMAPPED_{original}"] = original

print(f"Name to RXCUI mappings: {len(name_to_rxcui):,}")
print(f"Unique RXCUIs: {len(set(name_to_rxcui.values())):,}")

# Check deduplication improvement
unique_names_before = len(name_to_rxcui)
unique_rxcuis = len(set(name_to_rxcui.values()))
print(f"\nUnique drug names BEFORE RXCUI mapping: {unique_names_before:,}")
print(f"Unique identifiers AFTER RXCUI mapping: {unique_rxcuis:,}")
print(f"Additional duplicates merged: {unique_names_before - unique_rxcuis:,}")


# =============================================================================
# POST-PROCESS: Fix international name variants in RXCUI mapping
# =============================================================================
# RxNorm approximateTerm sometimes returns international spellings instead
# of standardizing to English generics. Fix known variants.

INTL_TO_ENGLISH = {
    'ATORVASTATINA': 'ATORVASTATIN',
    'AMLODIPINO': 'AMLODIPINE',
    'DICLOFENACO': 'DICLOFENAC',
    'DOXORUBICINA': 'DOXORUBICIN',
    'ROSUVASTATINA': 'ROSUVASTATIN',
    'SERTRALINA': 'SERTRALINE',
    'AMITRIPTILINA': 'AMITRIPTYLINE',
    'VENLAFAXINA': 'VENLAFAXINE',
    'QUETIAPINA': 'QUETIAPINE',
    'FLUOXETINUM': 'FLUOXETINE',
    'PAROXETINUM': 'PAROXETINE',
    'METFORMINUM': 'METFORMIN',
    'CETIRIZINA': 'CETIRIZINE',
    'CLOPIDOGRELUM': 'CLOPIDOGREL',
    'DILTIAZEMUM': 'DILTIAZEM',
    'BROMAZEPAMUM': 'BROMAZEPAM',
    'HYDROXYZINUM': 'HYDROXYZINE',
    'LOPERAMIDA': 'LOPERAMIDE',
    'METOCLOPRAMIDA': 'METOCLOPRAMIDE',
    'PROMETAZINA': 'PROMETHAZINE',
    'TRAZODONA': 'TRAZODONE',
    'TAMSULOSINA': 'TAMSULOSIN',
    'IVABRADINA': 'IVABRADINE',
    'CASPOFUNGINA': 'CASPOFUNGIN',
    'MICAFUNGINA': 'MICAFUNGIN',
    'CEFEPIMA': 'CEFEPIME',
    'CLEMASTINA': 'CLEMASTINE',
    'EPARINA': 'HEPARIN',
    'IDROMORFONE': 'HYDROMORPHONE',
    'NALOXONA': 'NALOXONE',
    'BEDAQUILINA': 'BEDAQUILINE',
    'ACIDO ALENDRONICO': 'ALENDRONIC ACID',
    'ACIDO FOLINICO': 'FOLINIC ACID',
    'CALCIO': 'CALCIUM',
    'PREDNISONA': 'PREDNISONE',
    'DEXAMETASONA': 'DEXAMETHASONE',
    'METOTREXATO': 'METHOTREXATE',
    'CICLOSPORINA': 'CYCLOSPORINE',
    'PROPANOLOL': 'PROPRANOLOL',
}

# Fix the rxcui_to_name mapping so standardized names use English
fixed_count = 0
for rxcui, name in rxcui_to_name.items():
    if name in INTL_TO_ENGLISH:
        rxcui_to_name[rxcui] = INTL_TO_ENGLISH[name]
        fixed_count += 1



print(f"International name variants fixed: {fixed_count} RXCUI mappings")
print(f"Variant dictionary size: {len(INTL_TO_ENGLISH)} known variants")



# -----------------------------------------------------------------------------
# Re-extract drug pairs using RXCUI
# -----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("Re-extracting drug pairs using RXCUI identifiers...")

# Apply RXCUI mapping to drug_clean
drug_clean['RXCUI'] = drug_clean['DRUGNAME_CLEAN'].map(name_to_rxcui)
drug_clean['DRUGNAME_STANDARD'] = drug_clean['RXCUI'].map(rxcui_to_name)

# Fill any missing standard names with original
drug_clean['DRUGNAME_STANDARD'] = drug_clean['DRUGNAME_STANDARD'].fillna(drug_clean['DRUGNAME_CLEAN'])

# Get unique RXCUIs per patient (this is the key improvement)
patient_drugs_std = drug_clean.groupby('PRIMARYID').agg({
    'RXCUI': lambda x: list(set(x)),
    'DRUGNAME_STANDARD': lambda x: list(set(x))
}).reset_index()

patient_drugs_std['NUM_DRUGS'] = patient_drugs_std['RXCUI'].apply(len)

print(f"Total patients: {len(patient_drugs_std):,}")
print(f"Patients with 2+ drugs: {(patient_drugs_std['NUM_DRUGS'] >= 2).sum():,}")

# Filter to patients with 2+ unique drugs
multi_drug_std = patient_drugs_std[patient_drugs_std['NUM_DRUGS'] >= 2].copy()

# Extract drug pairs using standardized names (but grouped by RXCUI)
print("\nExtracting drug pairs...")

drug_pairs_std = []
total = len(multi_drug_std)

for i, (idx, row) in enumerate(multi_drug_std.iterrows()):
    if i % 100000 == 0:
        print(f"  Processing: {i:,}/{total:,} ({100*i/total:.1f}%)")

    primaryid = row['PRIMARYID']
    drugs = sorted(row['DRUGNAME_STANDARD'])  # Use display names

    # Limit to max 20 drugs per patient
    if len(drugs) > 20:
        drugs = sorted(random.sample(drugs, 20))

    for pair in combinations(drugs, 2):
        drug_pairs_std.append((primaryid, pair[0], pair[1], f"{pair[0]} + {pair[1]}"))

print(f"  Processing: {total:,}/{total:,} (100%)")

drug_pairs_std_df = pd.DataFrame(drug_pairs_std, columns=['PRIMARYID', 'DRUG_A', 'DRUG_B', 'PAIR'])
print(f"\nTotal drug pairs extracted: {len(drug_pairs_std_df):,}")
print(f"Unique drug combinations: {drug_pairs_std_df['PAIR'].nunique():,}")

# -----------------------------------------------------------------------------
# Filter to combinations with 10+ occurrences
# -----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("Filtering to combinations with 10+ occurrences...")

pair_counts_std = drug_pairs_std_df['PAIR'].value_counts()
print(f"Pairs appearing 1 time: {(pair_counts_std == 1).sum():,}")
print(f"Pairs appearing 2-9 times: {((pair_counts_std >= 2) & (pair_counts_std < 10)).sum():,}")
print(f"Pairs appearing 10+ times: {(pair_counts_std >= 10).sum():,}")
print(f"Pairs appearing 100+ times: {(pair_counts_std >= 100).sum():,}")

# Keep only pairs with 10+ occurrences
frequent_pairs_std = pair_counts_std[pair_counts_std >= 10].index.tolist()
drug_pairs_std_filtered = drug_pairs_std_df[drug_pairs_std_df['PAIR'].isin(frequent_pairs_std)].copy()

# Merge with outcome data
drug_pairs_std_filtered = drug_pairs_std_filtered.merge(outcome_flags, on='PRIMARYID', how='inner')
print(f"Patients with outcome data retained: {drug_pairs_std_filtered['PRIMARYID'].nunique():,}")

print(f"\nAfter filtering to 10+ occurrences:")
print(f"  Drug pairs remaining: {len(drug_pairs_std_filtered):,}")
print(f"  Unique combinations: {drug_pairs_std_filtered['PAIR'].nunique():,}")
print(f"  Serious outcomes: {drug_pairs_std_filtered['SERIOUS'].sum():,}")

# Show top 20 most common drug pairs
print("\nTop 20 Most Common Drug Pairs (RXCUI Standardized):")
print(drug_pairs_std_filtered['PAIR'].value_counts().head(20))

# Save
drug_pairs_std_filtered.to_csv("../data/intermediate/FAERS_DRUG_PAIRS_RXNORM.csv", index=False)
print("\n Saved: FAERS_DRUG_PAIRS_RXNORM.csv")

Applying RxNorm standardization using RXCUI codes...
RxNorm mappings loaded: 64,552
Successfully mapped to RXCUI: 59,316
Name to RXCUI mappings: 64,552
Unique RXCUIs: 30,575

Unique drug names BEFORE RXCUI mapping: 64,552
Unique identifiers AFTER RXCUI mapping: 30,575
Additional duplicates merged: 33,977
International name variants fixed: 35 RXCUI mappings
Variant dictionary size: 39 known variants

Re-extracting drug pairs using RXCUI identifiers...
Total patients: 1,469,305
Patients with 2+ drugs: 556,666

Extracting drug pairs...
  Processing: 0/556,666 (0.0%)
  Processing: 100,000/556,666 (18.0%)
  Processing: 200,000/556,666 (35.9%)
  Processing: 300,000/556,666 (53.9%)
  Processing: 400,000/556,666 (71.9%)
  Processing: 500,000/556,666 (89.8%)
  Processing: 556,666/556,666 (100%)

Total drug pairs extracted: 14,925,815
Unique drug combinations: 2,213,209

Filtering to combinations with 10+ occurrences...
Pairs appearing 1 time: 1,245,842
Pairs appearing 2-9 times: 740,923
Pairs a

## Save intermediates for downstream notebooks

Persist `demo_all` and `outcome_flags` as CSVs so notebooks 03 and 06 can load them with the full patient denominator (not just the exposed subset).

In [ ]:
# Save in-memory DataFrames for downstream notebooks
demo_all.to_csv("../data/intermediate/FAERS_DEMO_ALL.csv", index=False)
print(f"Saved demo_all: {len(demo_all):,} rows -> ../data/intermediate/FAERS_DEMO_ALL.csv")

outcome_flags.to_csv("../data/intermediate/FAERS_OUTCOME_FLAGS.csv", index=False)
print(f"Saved outcome_flags: {len(outcome_flags):,} rows -> ../data/intermediate/FAERS_OUTCOME_FLAGS.csv")